# SkyPortal corpus — decision dossier (B)

This notebook is the methodology record for the sixteen normalisation decisions applied
by `scripts/skyportal/02_normalise.py`. For each one it measures the actual rows that made
the decision necessary, read fresh from `data/interim/skyportal_corpus/` only — not
imported from the script and not read from the corpus — so every figure here is one that
was visible at the moment the decision was taken. What each decision changed once applied
is measured in `C_normalisation.ipynb`; this notebook does not repeat that. It is written
for someone deciding whether to accept a decision, not someone checking that it was
applied.

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 250)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/interim/skyportal_corpus").is_dir())
INTERIM_ROOT = ROOT / "data/interim/skyportal_corpus"
TABLE_NAMES = ["sources", "comments", "photometry", "spectra", "followup_requests"]
EXPECTED_SHAPE = {"sources": (982, 114), "comments": (2950, 13), "photometry": (7968, 44),
                  "spectra": (1, 52), "followup_requests": (2359, 164)}

interim = {t: pd.read_parquet(INTERIM_ROOT / f"{t}.parquet") for t in TABLE_NAMES}
for name, frame in interim.items():
    if len(frame) == 0:
        raise ValueError(f"Table '{name}' loaded 0 rows")


def has_content(series):
    """True where a cell holds real content: '', '[]', '{}' and blanks are empty."""
    def alive(value):
        if value is None:
            return False
        if isinstance(value, str):
            return value.strip() not in {"", "[]", "{}"}
        if isinstance(value, float) and np.isnan(value):
            return False
        if isinstance(value, np.ndarray):
            return value.size > 0
        try:
            if pd.isna(value):
                return False
        except (TypeError, ValueError):
            pass
        return True
    return series.map(alive)


def canonical(value):
    """Canonicalise one cell for cross-copy comparison: sort JSON object keys."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    if isinstance(value, str) and value[:1] in "[{":
        try:
            return json.dumps(json.loads(value), sort_keys=True, ensure_ascii=False,
                               separators=(",", ":"))
        except (ValueError, TypeError):
            return value
    return value


print("tables loaded:")
for name, frame in interim.items():
    print(f"  {name:20s} rows={len(frame):5d} columns={frame.shape[1]:4d}")

controls = pd.DataFrame(
    [{"table": n, "expected_rows": EXPECTED_SHAPE[n][0], "observed_rows": interim[n].shape[0],
      "expected_cols": EXPECTED_SHAPE[n][1], "observed_cols": interim[n].shape[1],
      "status": "PASS" if interim[n].shape == EXPECTED_SHAPE[n] else "FAIL"} for n in TABLE_NAMES])
print("\nCONTROLS")
print(controls.to_string(index=False))
if (controls["status"] != "PASS").any():
    raise ValueError("control check failed:\n" + controls.to_string(index=False))

tables loaded:
  sources              rows=  982 columns= 114
  comments             rows= 2950 columns=  13
  photometry           rows= 7968 columns=  44
  spectra              rows=    1 columns=  52
  followup_requests    rows= 2359 columns= 164

CONTROLS
            table  expected_rows  observed_rows  expected_cols  observed_cols status
          sources            982            982            114            114   PASS
         comments           2950           2950             13             13   PASS
       photometry           7968           7968             44             44   PASS
          spectra              1              1             52             52   PASS
followup_requests           2359           2359            164            164   PASS


## Decisions 1 and 2 — one row per source

**Observed.** The interim `sources` table holds 982 rows but only 800 distinct
identifiers; 182 sources were returned by more than one listing profile.
**Why it matters.** Any per-source statistic computed directly on this table
double-counts every one of those 182 sources.
**Decided.** Collapse to one row per identifier; when the two copies genuinely
differ, keep the one carrying host-galaxy enrichment.
**Scope.** 182 sources are affected; after canonicalising JSON key order, only
1 of them actually differs between its copies, confined to the host-enrichment
columns.

In [2]:
print(f"interim sources rows: {len(interim['sources'])} | distinct ids: "
     f"{interim['sources']['id'].astype(str).str.strip().nunique()}")

multi = interim["sources"].assign(sid=interim["sources"]["id"].astype(str).str.strip())
counts = multi["sid"].value_counts()
multi_ids = sorted(counts[counts > 1].index)
print(f"multi-profile ids: {len(multi_ids)}")

example_id = multi_ids[0]
example_rows = multi[multi["sid"] == example_id][["id", "source_profile"]]
print(f"\nexample multi-profile source {example_id!r}, both interim rows:")
print(example_rows.to_string(index=False))

exclude = {"id", "source_profile", "source_file", "sid"}
compare_columns = [c for c in multi.columns if c not in exclude]
differing = {}
for source_id in multi_ids:
    group = multi[multi["sid"] == source_id]
    row_a, row_b = group.iloc[0], group.iloc[1]
    diffs = [c for c in compare_columns if canonical(row_a[c]) != canonical(row_b[c])]
    if diffs:
        differing[source_id] = diffs
print(f"\ngenuinely differ after canonicalising JSON key order: {len(differing)} of {len(multi_ids)}")

for source_id, columns in differing.items():
    group = multi[multi["sid"] == source_id]
    side_by_side = group.set_index("source_profile")[columns].T
    print(f"\nhost-family columns for {source_id!r}, side by side by profile:")
    print(side_by_side.to_string())

interim sources rows: 982 | distinct ids: 800
multi-profile ids: 182

example multi-profile source 'EP-241103_012438', both interim rows:
              id source_profile
EP-241103_012438             ep
EP-241103_012438   grandma_base

genuinely differ after canonicalising JSON key order: 1 of 182



host-family columns for 'INTEGRAL-GRB231115A', side by side by profile:
source_profile                 grandma_base   grb
host_offset                        8.908907   NaN
host.catalog_id                         1.0   NaN
host.created_at  2023-03-30T15:55:44.068376  None
host.name              CLU J095558.5+694049  None
host.modified    2023-03-30T15:55:44.068376  None
host.distmpc                        170.853   NaN
host.ra                          148.993804   NaN
host.dec                          69.680353   NaN
host.healpix           535275223798153280.0   NaN
host.id                          10194051.0   NaN


## Decision 3 — the duplicated follow-up requests

**Observed.** 20 ids in `followup_requests` appear on exactly two rows each; for
every one, `obj_id` is identical between the copies but `source_dir` differs.
**Why it matters.** Without a rule to pick one, 20 requests would be attributed
to whichever capture directory happened to load last — including a source
that never made them.
**Decided.** Keep the row where `source_dir` equals `obj_id`.
**Scope.** 20 duplicated ids, 40 rows; the rule resolves every one — exactly
one of each pair's two rows satisfies `source_dir == obj_id`.

In [3]:
fr = interim["followup_requests"]
dup_counts = fr["id"].value_counts()
dup_ids = sorted(dup_counts[dup_counts > 1].index)
print(f"duplicated ids: {len(dup_ids)}")

rows = []
for duplicate_id in dup_ids:
    group = fr[fr["id"] == duplicate_id]
    for _, row in group.iterrows():
        rows.append({"id": duplicate_id, "obj_id": row["obj_id"], "source_dir": row["source_dir"],
                     "source_dir_equals_obj_id": row["source_dir"] == row["obj_id"]})
duplicates = pd.DataFrame(rows)
obj_id_identical = duplicates.groupby("id")["obj_id"].nunique().eq(1).all()
print(f"obj_id identical within every duplicated id: {obj_id_identical}")
print(f"exactly one copy satisfies source_dir == obj_id, per id: "
     f"{(duplicates.groupby('id')['source_dir_equals_obj_id'].sum() == 1).all()}")
duplicates

duplicated ids: 20
obj_id identical within every duplicated id: True
exactly one copy satisfies source_dir == obj_id, per id: True


,id,obj_id,source_dir,source_dir_equals_obj_id
0,21009,EP240626A-FXT,EP240626A,False
1,21009,EP240626A-FXT,EP240626A-FXT,True
2,21010,EP240626A-FXT,EP240626A,False
3,21010,EP240626A-FXT,EP240626A-FXT,True
4,21011,EP240626A-FXT,EP240626A,False
5,21011,EP240626A-FXT,EP240626A-FXT,True
6,21016,EP240626A-FXT,EP240626A,False
7,21016,EP240626A-FXT,EP240626A-FXT,True
8,21019,EP240626A-FXT,EP240626A,False
9,21019,EP240626A-FXT,EP240626A-FXT,True


## Decisions 4 and 5 — whitespace

**Observed.** One source identifier carries a literal trailing tab; across the
five tables, 1,383 cells in twenty-two columns carry leading or trailing
whitespace that survived the raw capture. `photometry.instrument_name` records
367 rows as `'NUTTelA-TAO '` and zero as the clean spelling — nothing in this
capture currently collides with it, but nothing stops a future capture from
returning the clean form and counting it as a second instrument.
**Why it matters.** A dirty identifier fails to join against its clean
counterpart; a dirty label is one keystroke away from silently splitting a
single category in two.
**Decided.** Strip identifiers before any join (decision 4); strip every text
column afterward (decision 5).
**Scope.** 1,383 cells across 22 columns carry whitespace, including the
identifier column itself.

In [4]:
dirty_id = interim["sources"].loc[
    interim["sources"]["id"].astype(str) != interim["sources"]["id"].astype(str).str.strip(), "id"
].iloc[0]
print(f"identifier carrying a trailing tab: {dirty_id!r}")

dedup_key = interim["sources"]["id"].astype(str).str.strip()
sources_dedup = interim["sources"][~dedup_key.duplicated(keep="first")]

rows = []
for table_name in TABLE_NAMES:
    df = sources_dedup if table_name == "sources" else interim[table_name]
    for column in df.columns:
        if df[column].dtype != object:
            continue
        series = df[column]
        is_str = series.map(lambda v: isinstance(v, str))
        changed = is_str & (series != series.map(lambda v: v.strip() if isinstance(v, str) else v))
        n = int(changed.sum())
        if n:
            rows.append({"table": table_name, "column": column, "rows_affected": n})
whitespace = pd.DataFrame(rows).sort_values(["table", "rows_affected"], ascending=[True, False])
print(f"\n{len(whitespace)} columns affected, {whitespace['rows_affected'].sum()} cells total:")
print(whitespace.to_string(index=False))

instrument = interim["photometry"]["instrument_name"].astype(str)
spellings = pd.DataFrame([
    {"spelling": "NUTTelA-TAO (trailing space)", "rows": int((instrument == "NUTTelA-TAO ").sum())},
    {"spelling": "NUTTelA-TAO (clean)", "rows": int((instrument == "NUTTelA-TAO").sum())},
])
print("\ninstrument label, both spellings:")
spellings

identifier carrying a trailing tab: 'AT2023toh\t'



22 columns affected, 1383 cells total:
            table                     column  rows_affected
         comments                       text            551
followup_requests              allocation.pi            172
followup_requests                obj.summary            108
followup_requests                     obj_id             17
followup_requests                     obj.id             17
followup_requests    obj.tns_info.discoverer             17
followup_requests                 source_dir             17
followup_requests        obj.redshift_origin              8
followup_requests      obj.tns_info.reporter              6
followup_requests                     status              2
followup_requests allocation.instrument.name              1
       photometry            instrument_name            367
       photometry                     origin             30
       photometry               altdata.note             12
       photometry              altdata.meta1              6


,spelling,rows
0,NUTTelA-TAO (trailing space),367
1,NUTTelA-TAO (clean),0


## Decisions 6 and 7 — columns removed

**Observed.** 58 columns across the five tables carry no content in any row;
separately, `comments.resourceType` and `photometry.magsys` are both constant
across every row, holding a single value each.
**Why it matters.** An always-empty column adds nothing to keep; a constant
column might describe the data (worth keeping) or the endpoint that served it
(not part of the data at all).
**Decided.** Drop every always-empty column; drop `comments.resourceType`
specifically, because it names the API endpoint, not a property of a comment;
keep `photometry.magsys`, because a magnitude system is a property of the
measurement itself.
**Scope.** 58 empty columns across the five tables; the two constant columns
reach 2,950 rows (`comments.resourceType`) and 7,968 rows (`photometry.magsys`).

In [5]:
print("columns with no content in any row, per table:")
empty_by_table = {}
for table_name in TABLE_NAMES:
    df = interim[table_name]
    empty = [c for c in df.columns if not has_content(df[c]).any()]
    empty_by_table[table_name] = empty
    print(f"  {table_name:20s} {len(empty)}")

print(f"\nfull list for 'sources' ({len(empty_by_table['sources'])} columns):")
print(sorted(empty_by_table["sources"]))

constant = pd.DataFrame([
    {"table": "comments", "column": "resourceType",
     "single_value": interim["comments"]["resourceType"].iloc[0],
     "rows": len(interim["comments"]), "describes": "the API endpoint that returned the record"},
    {"table": "photometry", "column": "magsys",
     "single_value": interim["photometry"]["magsys"].iloc[0],
     "rows": len(interim["photometry"]), "describes": "a property of the measurement itself"},
])
constant

columns with no content in any row, per table:
  sources              25
  comments             1
  photometry           2
  spectra              10


  followup_requests    20

full list for 'sources' (25 columns):
['altdata', 'dec_dis', 'dec_err', 'detect_photometry_count', 'dist_nearest_source', 'e_mag_nearest_source', 'host.a', 'host.alt_name', 'host.distmpc_unc', 'host.mag_fuv', 'host.mag_nuv', 'host.mag_w1', 'host.mag_w2', 'host.mag_w3', 'host.mag_w4', 'host.magk', 'host.redshift', 'host.redshift_error', 'host.sfr_w4', 'mag_nearest_source', 'mpc_name', 'ra_dis', 'ra_err', 'score', 'tns_info.end_prop_period']


,table,column,single_value,rows,describes
0,comments,resourceType,sources,2950,the API endpoint that returned the record
1,photometry,magsys,ab,7968,a property of the measurement itself


## Decision 8 — sparse columns retained

**Observed.** Content coverage across the 114 columns of the flattened
`sources` table (deduplicated to its 800 distinct identities) ranges from 0%
to 100%, with a median around 7%; `redshift` sits at 58 of 800 sources, none
of which is empty where present.
**Why it matters.** A blanket rule dropping low-coverage columns would remove
real astronomical values along with genuinely empty ones.
**Decided.** Keep every column that carries at least one real value, however
rare, and document its coverage rather than discard it.
**Scope.** `redshift` reaches 58 of 800 sources; the same argument applies to
every column at any coverage above zero.

In [6]:
dedup_key = interim["sources"]["id"].astype(str).str.strip()
sources_flat = interim["sources"][~dedup_key.duplicated(keep="first")]
coverage = pd.Series({c: 100 * has_content(sources_flat[c]).sum() / len(sources_flat)
                      for c in sources_flat.columns}).sort_values()
print(f"content coverage across the {len(coverage)} columns of the flattened sources "
     f"table ({len(sources_flat)} distinct identities):")
print(coverage.describe().to_string())

redshift_present = has_content(sources_flat["redshift"])
redshift = sources_flat.loc[redshift_present, ["id", "redshift"]]
print(f"\n'redshift': {len(redshift)} of {len(sources_flat)} sources carry a value")
redshift.sort_values("redshift").reset_index(drop=True)

content coverage across the 114 columns of the flattened sources table (800 distinct identities):
count    114.000000
mean      22.382675
std       36.854114
min        0.000000
25%        0.125000
50%        7.250000
75%       14.250000
max      100.000000

'redshift': 58 of 800 sources carry a value


,id,redshift
0,SN2023wrk,0.0230
1,AT2024hdl,0.0416
2,ZTF25aciopbi,0.0461
3,GOTO23baj,0.0560
4,EP250207b,0.0820
5,ZTF23abidzvf,0.1500
6,EP250304a,0.2000
7,GRB-250424_065229,0.3100
8,250206dm-DDOTI-1,0.4500
9,2026owq,0.4730


## Decision 9 — status

**Observed.** `followup_requests.status` holds 187 distinct strings; grouping
by the longest matching prefix collapses all but 26 of them into seven
categories, and several of those 26 describe an outcome (a photometry commit,
an image posted) rather than a request's lifecycle stage.
**Why it matters.** Left as free text, `status` cannot answer how many
requests failed, because failure reasons and timestamps are concatenated
straight into the label.
**Decided.** Add `status_normalised` with the seven lifecycle prefixes plus
`processing_result` for the rest, and keep `status` unchanged alongside it.
**Scope.** All 2,359 rows of `followup_requests`; every one of the 187 strings
resolves to exactly one of eight values.

In [7]:
status = interim["followup_requests"]["status"].astype(str).str.strip()
print(f"distinct status strings: {status.nunique()}")

prefixes = ["failed to submit", "submitted for", "submitted", "deleted", "complete", "rejected", "pending"]
ordered = sorted(prefixes, key=len, reverse=True)
assigned = pd.Series(False, index=status.index)
prefix_of = pd.Series("processing_result", index=status.index)
for prefix in ordered:
    match = status.str.startswith(prefix) & ~assigned
    prefix_of[match] = prefix
    assigned |= match

summary = (pd.DataFrame({"prefix": prefix_of, "status": status})
          .groupby("prefix")["status"].agg(rows="size", distinct_strings="nunique")
          .reindex(ordered + ["processing_result"]))
print("\nrows and distinct strings per prefix:")
print(summary.to_string())

print("\nthree full status values (100 chars) with error text in the label:")
for value in status[status.str.len() > 60].drop_duplicates().head(3):
    print(f"  {value[:100]}")

print("\nvalues describing a processing result rather than a request state:")
status[prefix_of == "processing_result"].value_counts()

distinct status strings: 187

rows and distinct strings per prefix:
                   rows  distinct_strings
prefix                                   
failed to submit   1028                10
submitted for       133               129
submitted           948                27
complete             40                 1
rejected             66                 8
deleted             103                 1
pending              15                 1
processing_result    26                10

three full status values (100 chars) with error text in the label:
  failed to submit: HTTPSConnectionPool(host='trt.narit.or.th', port=443): Max retries exceeded with u
  No photometry available: No records (epochs) returned by database query
  submitted for 2025-03-27 23:11:46: use retrieve to check status

values describing a processing result rather than a request state:


status
Photometry committed to database                                           5
No photometry available: No records (epochs) returned by database query    5
No photometry to commit to database                                        4
16 images posted as comment                                                3
10 images posted as comment                                                3
8 images posted as comment                                                 2
25 images posted as comment                                                1
24 images posted as comment                                                1
7 images posted as comment                                                 1
1 images posted as comment                                                 1
Name: count, dtype: int64

## Decisions 10 and 15 — time

**Observed.** Every datetime-shaped column in the flattened tables arrives as
plain text with no timezone offset, and three `followup_requests` columns mix
three different on-disk formats for the same field. Only `created_at` reaches
100% coverage in all five tables; `mjd` — the observation instant — precedes
its own row's `created_at` — the recording instant — by a median of about 34
days.
**Why it matters.** A column that is untyped, timezone-less and inconsistently
formatted cannot be compared or truncated safely, and one that dates the sky
event rather than the moment SkyPortal recorded it cannot anchor a query about
what was known at a past instant — most such columns are too sparse to anchor
anything at all.
**Decided.** Type every ISO column as explicit UTC; use `created_at` — never
`mjd`, `t0`, or the other observation-time columns — as the causal truncation
anchor.
**Scope.** Every candidate datetime column across the five tables, none typed
and none timezone-aware; `created_at` alone reaches 100% coverage in every
table against a median 34-day gap to `mjd`.

In [8]:
ISO_RE = re.compile(r"^\d{4}-\d{2}-\d{2}([T ]\d{2}:\d{2}(:\d{2}(\.\d+)?)?)?$")
FORMAT_RE = re.compile(
    r"^(?P<date>\d{4}-\d{2}-\d{2})"
    r"(?:(?P<sep>[T ])(?P<time>\d{2}:\d{2}:\d{2})(?P<frac>\.\d+)?)?"
    r"(?P<tz>Z|[+-]\d{2}:?\d{2})?$")


def classify_format(value):
    match = FORMAT_RE.match(value)
    date_part = "date-only" if not match.group("sep") else f"sep={match.group('sep')!r}"
    frac_part = "frac" if match.group("frac") else "no-frac"
    tz_part = "tz" if match.group("tz") else "no-tz"
    return f"{date_part} {frac_part} {tz_part}"


rows = []
for table_name in TABLE_NAMES:
    df = interim[table_name]
    for column in df.columns:
        if df[column].dtype != object:
            continue
        series = df[column]
        present = series[has_content(series)]
        if present.empty:
            continue
        text = present.astype(str).str.strip()
        if not text.map(lambda v: bool(ISO_RE.match(v))).all():
            continue
        formats = sorted(text.map(classify_format).unique())
        rows.append({"table": table_name, "column": column, "dtype": str(series.dtype),
                     "coverage_pct": round(100 * len(present) / len(df), 1), "formats": formats})

datetimes = pd.DataFrame(rows)
print(f"candidate datetime columns: {len(datetimes)}, all dtype=object: "
     f"{(datetimes['dtype'] == 'object').all()}")
any_tz = any(fmt.endswith(" tz") for formats in datetimes["formats"] for fmt in formats)
print(f"any column carrying an explicit UTC offset: {any_tz}")

mixed = datetimes[datetimes["formats"].map(len) > 1]
print(f"\ncolumns with more than one on-disk format ({len(mixed)}):")
for _, r in mixed.iterrows():
    print(f"  {r['table']}.{r['column']}: {r['formats']}")

print("\ncreated_at coverage against every other datetime column, per table:")
coverage = datetimes[["table", "column", "coverage_pct"]].sort_values(
    ["table", "coverage_pct"], ascending=[True, False])
print(coverage.to_string(index=False))

photometry = interim["photometry"]
MJD_EPOCH = pd.Timestamp("1858-11-17", tz="UTC")
observed = MJD_EPOCH + pd.to_timedelta(photometry["mjd"], unit="D")
created = pd.to_datetime(photometry["created_at"], format="ISO8601", utc=True)
gap_days = (observed - created).dt.total_seconds() / 86400
print(f"\nphotometry: mjd (observation) minus created_at (recording), in days -- "
     f"{gap_days.notna().sum()} rows:")
gap_days.describe()

candidate datetime columns: 30, all dtype=object: True
any column carrying an explicit UTC offset: False

columns with more than one on-disk format (3):
  followup_requests.payload.end_date: ['date-only no-frac no-tz', "sep=' ' frac no-tz", "sep=' ' no-frac no-tz"]
  followup_requests.payload.start_date: ['date-only no-frac no-tz', "sep=' ' frac no-tz", "sep=' ' no-frac no-tz"]
  followup_requests.payload.date: ["sep='T' frac no-tz", "sep='T' no-frac no-tz"]

created_at coverage against every other datetime column, per table:
            table                           column  coverage_pct
         comments                       created_at         100.0
         comments                         modified         100.0
followup_requests                         modified         100.0
followup_requests                       created_at         100.0
followup_requests                   obj.created_at         100.0
followup_requests                     obj.modified         100.0
followup_requ

count     7968.000000
mean      -108.692015
std       1143.471248
min     -58549.509624
25%       -117.435208
50%        -34.217165
75%         -3.898844
max       1452.327931
dtype: float64

## Decisions 11 and 12 — values that cannot be true

**Observed.** Ten photometry rows carry `limiting_mag = -1.0`, a value
magnitudes cannot take; five rows carry an `mjd` that, converted to a calendar
date, falls in 1864 or in mid-2027 — before the raw capture existed and after
it was taken. Fourteen further rows carry an `mjd` later than their own
`created_at`, but by less than a day, consistent with normal upload lag.
**Why it matters.** A sentinel or an impossible date poisons any statistic
computed over the column unless it is recognised as not-a-value.
**Decided.** Null `limiting_mag == -1.0`; null `mjd` outside [55000, 61250]
and flag it; leave the fourteen small, plausible gaps alone.
**Scope.** 10 rows reached by the sentinel rule, 5 by the out-of-range rule;
the fourteen plausible-gap rows fall outside both and stay untouched.

In [9]:
photometry = interim["photometry"]
sentinel = photometry[photometry["limiting_mag"] == -1.0]
print(f"rows with limiting_mag == -1.0 (sentinel): {len(sentinel)}")
print(sentinel[["id", "obj_id", "limiting_mag"]].to_string(index=False))

MJD_EPOCH = pd.Timestamp("1858-11-17")
out_of_range = photometry[(photometry["mjd"] < 55000) | (photometry["mjd"] > 61250)].copy()
out_of_range["mjd_as_date"] = MJD_EPOCH + pd.to_timedelta(out_of_range["mjd"], unit="D")
print(f"\nrows with mjd outside [55000, 61250]: {len(out_of_range)}")
print(out_of_range[["id", "obj_id", "mjd", "mjd_as_date", "created_at"]].to_string(index=False))

observed = MJD_EPOCH + pd.to_timedelta(photometry["mjd"], unit="D")
created = pd.to_datetime(photometry["created_at"])
gap_days = (observed - created).dt.total_seconds() / 86400
plausible = photometry[(gap_days > 0) & (gap_days < 1) & ~photometry.index.isin(out_of_range.index)]
print(f"\nrows where mjd exceeds created_at by less than a day (left alone): {len(plausible)}")

rows with limiting_mag == -1.0 (sentinel): 10
   id     obj_id  limiting_mag
54941    2025azm          -1.0
54942    2025azn          -1.0
43779  GRB241127          -1.0
43780  GRB241127          -1.0
55004 GRB250207A          -1.0
54998 GRB250207A          -1.0
54999 GRB250207A          -1.0
55000 GRB250207A          -1.0
55002 GRB250207A          -1.0
54997 GRB250207A          -1.0

rows with mjd outside [55000, 61250]: 5
   id       obj_id         mjd                   mjd_as_date                 created_at
31672   GRB240911A  2024.00000 1864-06-02 00:00:00.000000000 2024-09-20T12:09:57.173028
31674   GRB240911A  2024.00000 1864-06-02 00:00:00.000000000 2024-09-20T12:13:51.494958
31673   GRB240911A  2024.00000 1864-06-02 00:00:00.000000000 2024-09-20T12:10:49.896550
 2999 ZTF23aaptsuy 61582.18280 2027-06-26 04:23:13.920000198 2023-07-04T21:08:04.366377
 3000 ZTF23aaptsuy 61582.21003 2027-06-26 05:02:26.592000172 2023-07-04T21:10:13.332703

rows where mjd exceeds created_at by less t

## Decision 13 — coordinates at exactly zero

**Observed.** Three `sources.ra` values, one `sources.dec` value, and
seventeen `followup_requests.obj.ra`/`obj.dec` pairs sit at exactly 0.0.
**Why it matters.** 0.0 is a legitimate point on the sky, and it is also the
default a missing coordinate falls back to; nothing in the stored value tells
the two apart.
**Decided.** Add a boolean flag column per coordinate marking exactly 0.0;
leave the coordinate value itself untouched.
**Scope.** 3 rows of `sources.ra`, 1 of `sources.dec`, and 17 rows of
`followup_requests` (`obj.ra` and `obj.dec` together, all on a single
`obj_id`).

In [10]:
dedup_key = interim["sources"]["id"].astype(str).str.strip()
sources_flat = interim["sources"][~dedup_key.duplicated(keep="first")]
followup = interim["followup_requests"]

print("rows with a coordinate exactly at 0.0, flattened tables:")
print(f"  sources.ra                 {int((sources_flat['ra'] == 0.0).sum())}")
print(f"  sources.dec                {int((sources_flat['dec'] == 0.0).sum())}")
print(f"  followup_requests.obj.ra   {int((followup['obj.ra'] == 0.0).sum())}")
print(f"  followup_requests.obj.dec  {int((followup['obj.dec'] == 0.0).sum())}")

zero_sources = sources_flat[(sources_flat["ra"] == 0.0) | (sources_flat["dec"] == 0.0)]
print("\nsources rows carrying a zero coordinate:")
print(zero_sources[["id", "ra", "dec"]].to_string(index=False))

zero_followup = followup[(followup["obj.ra"] == 0.0) | (followup["obj.dec"] == 0.0)]
print(f"\nfollowup_requests rows carrying a zero coordinate: {len(zero_followup)}, "
     f"distinct obj_id: {sorted(zero_followup['obj_id'].unique())}")

rows with a coordinate exactly at 0.0, flattened tables:
  sources.ra                 3
  sources.dec                1
  followup_requests.obj.ra   17
  followup_requests.obj.dec  17

sources rows carrying a zero coordinate:
               id  ra     dec
S251112cm_TCH2882 0.0 -31.818
S251112cm_TCH2716 0.0 -33.636
        S230628ax 0.0   0.000

followup_requests rows carrying a zero coordinate: 17, distinct obj_id: ['S230628ax']


## Decision 14 — provenance left as written

**Observed.** `photometry.origin` holds 75 distinct values in the interim
table: URLs, GCN circular numbers, raw coordinate pairs, pipeline task ids,
and free text. `'GCN'` and `'zmaksut'` each appear twice, once clean and once
with a trailing space, at very different counts.
**Why it matters.** These are hand-written provenance notes, not a closed
vocabulary — an equivalence rule broad enough to unify `'GCN'` with
`'GCN Circular 39737'` would also wrongly unify unrelated notes that happen
to share a word.
**Decided.** Leave `origin` as written, beyond the whitespace stripping
already applied everywhere (decision 5).
**Scope.** 75 distinct values narrow to 73 once whitespace is stripped; only
the two pairs above merge, the rest stand alone.

In [11]:
photometry = interim["photometry"]
origin = photometry["origin"].dropna()
origin = origin[origin.astype(str).str.strip() != ""]
print(f"distinct origin values (raw): {origin.nunique()}")

stripped = origin.astype(str).str.strip()
print(f"distinct origin values (whitespace stripped): {stripped.nunique()}")

merging = origin.astype(str).groupby(stripped).apply(lambda s: sorted(set(s)))
merging = {k: v for k, v in merging.items() if len(v) > 1}
print("\nvalues that merge once stripped:")
for clean, variants in merging.items():
    for variant in variants:
        print(f"  {variant!r}: {int((origin.astype(str) == variant).sum())} rows")

print("\nfull inventory (raw), value and row count, most common first:")
origin.value_counts()

distinct origin values (raw): 75
distinct origin values (whitespace stripped): 73

values that merge once stripped:
  'GCN': 975 rows
  'GCN ': 22 rows
  'zmaksut': 359 rows
  'zmaksut ': 8 rows

full inventory (raw), value and row count, most common first:


origin
fp                                                      2786
None                                                    2135
stdview                                                  995
GCN                                                      975
zmaksut                                                  359
stdweb                                                   289
weikang                                                  135
UVOT                                                      40
Alain                                                     36
Baader                                                    35
GCN                                                       22
XRT                                                       17
GRANDMA-GCN                                                9
STDPipe analysis                                           8
Broens-init                                                8
zmaksut                                                    8
GCN 41162        

## Decision 16 — field history

**Observed.** `redshift_history` holds 83 dated entries across 60 sources;
`summary_history` holds 1,274 across 256, including entries whose value is
null. 141 of the 256 summary arrays are not stored in the same order as their
own timestamps — some run newest-first.
**Why it matters.** Reading the array's first element as "the current value"
silently inverts the history for those 141 sources, and dropping the null
entries would hide that a field was explicitly cleared, not just left unset.
**Decided.** Expand both columns into `source_field_history`, one row per
entry, keeping null values as deletion events and recording each entry's
original array position in `entry_index`.
**Scope.** 1,357 entries from 266 distinct sources (83 redshift across 60,
1,274 summary across 256); 28 of them are deletion events with a null value.

In [12]:
HISTORY_PARENT = {"redshift_history": "redshift", "summary_history": "summary"}
HISTORY_VALUE = {"redshift_history": "value", "summary_history": "summary"}

dedup_key = interim["sources"]["id"].astype(str).str.strip()
unique_sources = interim["sources"][~dedup_key.duplicated(keep="first")]

history_rows = []
for field, parent in HISTORY_PARENT.items():
    value_key = HISTORY_VALUE[field]
    carrying = unique_sources[unique_sources[field].notna()]
    for _, row in carrying.iterrows():
        for index, entry in enumerate(json.loads(row[field])):
            history_rows.append({"source_id": row["id"], "field": parent, "entry_index": index,
                                 "value": entry.get(value_key), "value_is_null": entry.get(value_key) is None,
                                 "set_at_utc": entry.get("set_at_utc"),
                                 "set_by_user_id": entry.get("set_by_user_id")})
history = pd.DataFrame(history_rows)

for field in ["redshift", "summary"]:
    rows = history[history["field"] == field]
    print(f"{field}: {len(rows)} entries across {rows['source_id'].nunique()} sources")

deletion = history[history["value_is_null"]].iloc[0]
print(f"\ndeletion events (value is null): {int(history['value_is_null'].sum())} total")
print("one such entry:")
print(deletion[["source_id", "field", "entry_index", "value", "value_is_null",
               "set_at_utc", "set_by_user_id"]].to_string())

chronological = history.sort_values(["source_id", "field", "set_at_utc"], kind="stable")
backwards = chronological.groupby(["source_id", "field"])["entry_index"].apply(
    lambda s: list(s) != sorted(s))
total_pairs = len(chronological.groupby(["source_id", "field"]))
print(f"\n(source, field) pairs where array order disagrees with chronological order: "
     f"{int(backwards.sum())} of {total_pairs}")
print(backwards[backwards].index.get_level_values("field").value_counts().to_string())

example_id, example_field = list(backwards[backwards].index)[0]
sequence = chronological[(chronological["source_id"] == example_id)
                         & (chronological["field"] == example_field)]
print(f"\n{example_id!r} ({example_field}), entry_index in chronological order "
     f"(array stored newest-first): {sequence['entry_index'].tolist()}")

redshift: 83 entries across 60 sources
summary: 1274 entries across 256 sources

deletion events (value is null): 28 total
one such entry:
source_id                         EP-250704_081653
field                                     redshift
entry_index                                      1
value                                         None
value_is_null                                 True
set_at_utc        2026-06-02T13:52:56.379891+00:00
set_by_user_id                                 137

(source, field) pairs where array order disagrees with chronological order: 141 of 316
field
summary    141

'2024abfo' (summary), entry_index in chronological order (array stored newest-first): [1, 0]


## What this record establishes

Each of the sixteen decisions appears here with the data that motivated
it, measured independently of the code that applies it. None was taken
by convention or by analogy: every one answers to something observable
in the flattened tables.

Three kinds of justification run through this record. Some decisions
correct data that cannot be true — an observation dated 2027 recorded in
2023, a limiting magnitude of -1.0. Others make explicit what was
implicit: 187 request statuses reduce to eight, and `entry_index`
preserves the original array position because in 141 sources that order
is not chronological. One, decision 13, does no more than record an
ambiguity the data cannot resolve: 0.0 is a valid sky coordinate and
also what a system writes when it has none.

Two decisions consist of not intervening. Sparse columns are kept
because sparse is not empty: redshift is present on 58 of the 800
sources and is a real value where it appears. And `origin` is left as
written, with its URLs, circular numbers and bare coordinates, because
these are hand-written notes and any equivalence rule would be right in
some cases and wrong in others.